# Chapter 12: Status Checks, Check Runs & Commit Statuses (Reference)

## Learning Objectives

- Name the status and conclusion vocabularies for the Checks API
- Publish a check run and read back its id/name/status/conclusion
- Compare the Commit Status API and the Checks API
- Explain why a job rename can silently orphan a required check

## Setup

The next cell sets up reproducibility, the `PRA_MODE` toggle, and inserts the repo root onto `sys.path`. You should see `PRA_MODE = 'fixture'` printed by default.

In [1]:
import os
import random
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pr_automerge").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

RANDOM_STATE: int = 42
random.seed(RANDOM_STATE)

PRA_MODE = os.environ.get("PRA_MODE", "fixture")
PRA_REPO = os.environ.get("PRA_REPO", "")

if PRA_MODE == "live":
    assert PRA_REPO, "Set PRA_REPO=owner/name to run against a real repo"

print(f"PRA_MODE = {PRA_MODE!r}")

PRA_MODE = 'fixture'


## 1. The Vocabulary

The next cell prints the `status` and `conclusion` vocabularies. You should see three status values and seven conclusion values.

In [2]:
from labs.lab_12_check_runs import CHECK_RUN_STATUSES, CHECK_RUN_CONCLUSIONS

print(f"status values:     {CHECK_RUN_STATUSES}")
print(f"conclusion values: {CHECK_RUN_CONCLUSIONS}")

status values:     ['queued', 'in_progress', 'completed']
conclusion values: ['success', 'failure', 'neutral', 'cancelled', 'skipped', 'timed_out', 'action_required']


## 2. Publish a Check Run

The next cell publishes a `gate3-risk-score` check run. You should see its returned `id`, `name`, and `status`/`conclusion` pair.

In [3]:
from labs.lab_12_check_runs import publish_check_run

result = publish_check_run(
    "example/example",
    "a1b2c3d4e5f60718293a4b5c6d7e8f9012345678",
    "gate3-risk-score",
    conclusion="success",
    title="risk=66.0 <= threshold=70",
    summary="Gate 3 passed.",
)
print(f"id: {result['id']}")
print(f"name: {result['name']}")
print(f"status/conclusion: {result['status']}/{result['conclusion']}")

id: 900123456
name: gate3-risk-score
status/conclusion: completed/success


## 3. Commit Status API vs Checks API

The next cell prints the comparison table. You should see the Checks API win on every row except nothing -- it's strictly richer, which is why this repo uses it exclusively.

In [4]:
from labs.lab_12_check_runs import commit_status_vs_check_run

for row in commit_status_vs_check_run():
    print(f"{row['property']:<28} status={row['commit_status']}")
    print(f"{'':<28} check_run={row['check_run']}")

Output detail                status=One state + one short description string
                             check_run=Title, Markdown summary, annotations on specific lines
Required-check matching      status=By context string
                             check_run=By check name string -- same matching mechanism, different field
Re-runnable from the UI      status=No
                             check_run=Yes, if published by a GitHub App
This repo's usage            status=Not used
                             check_run=Used by every gate workflow


## Takeaways & Next Steps

This notebook's takeaway is the `name` field printed in Section 2 -- that exact string is what branch protection matches against as a required check.

In [5]:
print("Re-run this notebook with PRA_MODE=live to publish a real check run.")

Re-run this notebook with PRA_MODE=live to publish a real check run.


---

📖 **Reading companion:** [Chapter 12: Status Checks, Check Runs & Commit Statuses](../learning_modules/chapter_12_status_checks.md)
